# Лабораторная работа №1. Нормализация текста

**Курс:** «Синтез речи», Университет ИТМО, 2026  
**Корпус:** RUSLAN  

В блокноте приведены постановка задачи, описание входных данных и признаков, метод решения, разведочный анализ, оценка фильтра и результаты обработки корпуса. Основная логика вынесена в отдельные модули, чтобы её можно было тестировать и использовать в следующих лабораторных работах.

## 1. Постановка задачи

Цель работы — подготовить текстовую часть корпуса RUSLAN для обучения системы синтеза речи, не нарушив соответствие между текстом и аудио. Для этого необходимо:

1. исследовать текстовые метаданные и определить типы ненормализованных данных;
2. реализовать классификатор, отличающий нормализованные фразы от ненормализованных;
3. реализовать консервативный текстовый нормализатор;
4. применить сначала нормализатор, затем фильтр ко всему корпусу;
5. сохранить результат в формате `id | исходный текст | нормализованный текст`;
6. проверить качество классификатора и контракт итогового файла.

Позитивный класс классификатора — текст, который можно буквально сопоставить с произнесённой фразой (`is_normalized = 1`). Требуемая F1-мера на открытом наборе — не менее 0,90. После обработки должно сохраниться не менее 16 000 фраз.

## 2. Входные данные и признаки

Исходные данные представлены корпусом RUSLAN: 22 200 аудиозаписей одного диктора и таблицей метаданных. Каждая строка исходной таблицы содержит два поля:

- `id` — идентификатор фразы и имя соответствующего WAV-файла;
- `text` — исходная расшифровка речи.

Фильтр использует интерпретируемые символьные признаки:

- наличие цифр и латинских букв;
- технические и именуемые символы;
- сокращения, инициалы и аббревиатуры;
- междометия и текстовые смайлы;
- повторная или ошибочная пунктуация;
- неправильные пробелы;
- непарные кавычки и скобки;
- невидимые и управляющие Unicode-символы;
- несоответствие нормальной форме Unicode NFC.

Эти признаки выбраны потому, что они непосредственно указывают на возможное расхождение между записью и произношением.

In [1]:
from collections import Counter
from pathlib import Path
import csv
import subprocess
import sys
import unicodedata
import wave

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'labs' / 'lab1_text' / 'text_filter.py').is_file():
            return candidate
    raise FileNotFoundError('Запустите блокнот из клонированного репозитория tts_labs_itmo')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
LAB_DIR = PROJECT_ROOT / 'labs' / 'lab1_text'
DATA_DIR = PROJECT_ROOT / 'data'
SOURCE_METADATA = DATA_DIR / 'RUSLAN' / 'metadata_RUSLAN_22200.csv'
NORMALIZED_METADATA = DATA_DIR / 'metadata_RUSLAN_22200_normalized.csv'
WAV_DIR = DATA_DIR / 'RUSLAN' / 'RUSLAN'
DEV_SET = LAB_DIR / 'data' / 'dev_sentences.csv'

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from analyze_ruslan import PATTERNS, analyze
from preprocess_ruslan import preprocess
from text_filter import TextFilter
from text_normalizer import TextNormalizer

print('Корень проекта:', PROJECT_ROOT)
print('Исходные метаданные:', SOURCE_METADATA)
print('Метаданные найдены:', SOURCE_METADATA.is_file())

Корень проекта: C:\work projects\ITMO\tts_labs_itmo
Исходные метаданные: C:\work projects\ITMO\tts_labs_itmo\data\RUSLAN\metadata_RUSLAN_22200.csv
Метаданные найдены: True


## 3. Разведочный анализ

Категории могут пересекаться, поэтому их количества нельзя суммировать. Анализ реализован в `analyze_ruslan.py`; ниже он запускается непосредственно из блокнота.

In [2]:
total, counts, examples = analyze(SOURCE_METADATA, example_limit=2)
eda_rows = [
    {
        'признак': name,
        'строк': counts.get(name, 0),
        'доля, %': round(counts.get(name, 0) / total * 100, 3),
        'примеры': ' / '.join(examples.get(name, [])),
    }
    for name in PATTERNS
]
eda_df = pd.DataFrame(eda_rows)
print('Всего строк:', total)
display(eda_df)

ax = eda_df.set_index('признак')['строк'].plot.bar(figsize=(10, 4), title='Формальные признаки в RUSLAN')
ax.set_ylabel('Количество строк')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Всего строк: 22200


,признак,строк,"доля, %",примеры
0,digits,4,0.018,Брат разъезжал по отдаленным лагерным точкам. ...
1,latin_letters,0,0.000,
2,technical_symbols,5,0.023,Я на букву О /библиография к Окуджаве/. / А Ле...
3,named_symbols,0,0.000,
4,likely_abbreviations,177,0.797,"А пиши ты, дурень, буквами А, Б, В. / О РАССКА..."
5,repeated_punctuation,621,2.797,"За что же моя рядовая, честная, единственная с..."
6,unusual_quotes,57,0.257,Мечтаем получить от вас рецензию.” / Единствен...
7,invisible_characters,0,0.000,
8,emoticons,0,0.000,


C:\Users\mrkaz\AppData\Local\Temp\ipykernel_20940\2320026779.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Основные результаты анализа полного корпуса:

| Категория | Строк | Доля |
|---|---:|---:|
| Цифры | 4 | 0,02% |
| Латинские буквы | 0 | 0,00% |
| Технические символы | 5 | 0,02% |
| Сокращения и инициалы по формальному шаблону | 177 | 0,80% |
| Повторная или необычная пунктуация | 621 | 2,80% |
| Необычные типографские кавычки | 57 | 0,26% |
| Невидимые символы и смайлы | 0 | 0,00% |

Большинство изменений связано не с произносимыми словами, а с пробелами и типографикой.

## 4. Метод решения

Обработка выполняется последовательно:

```text
исходный текст → консервативная нормализация → фильтрация → итоговые метаданные
```

`TextNormalizer` исправляет только непроизносимые типографические признаки: Unicode NFC, пробелы, варианты дефиса и тире, кавычки, повторную пунктуацию и непарные знаки. Регистр, буква `ё` и знак ударения `U+0301` сохраняются.

`TextFilter` отклоняет неоднозначные случаи: числа, латиницу, сокращения, инициалы, аббревиатуры и технические символы. Они не раскрываются автоматически, потому что письменную форму нельзя надёжно восстановить в реально произнесённую без прослушивания.

Выбран набор правил, а не обучаемая модель: dev-набор содержит только 300 строк, признаки имеют явную символьную природу, а правила дают объяснимую причину каждого отклонения.

In [3]:
normalizer = TextNormalizer()
normalization_examples = [
    '  Мой отец – рядовым…  ',
    'Белла.  Мы зашли',
    '„повестью в диалогах”',
    'шатается по бульварам?..',
]
pd.DataFrame({
    'исходный текст': normalization_examples,
    'результат': [normalizer.normalize(text) for text in normalization_examples],
})

,исходный текст,результат
0,Мой отец – рядовым…,Мой отец — рядовым…
1,Белла. Мы зашли,Белла. Мы зашли
2,„повестью в диалогах”,"""повестью в диалогах"""
3,шатается по бульварам?..,шатается по бульварам?…


In [4]:
text_filter = TextFilter()
filter_examples = [
    'Обычная нормализованная фраза.',
    'В 1995 г. произошло событие.',
    'Александрову Г. П. передали письмо.',
    'КПСС приняла решение.',
]
pd.DataFrame([
    {
        'текст': text,
        'is_normalized': text_filter.filter(text),
        'причины отклонения': ', '.join(text_filter.rejection_reasons(text)) or '—',
    }
    for text in filter_examples
])

,текст,is_normalized,причины отклонения
0,Обычная нормализованная фраза.,1,—
1,В 1995 г. произошло событие.,0,"digits, abbreviations"
2,Александрову Г. П. передали письмо.,0,"abbreviations, initials"
3,КПСС приняла решение.,0,"known_acronyms, consonant_acronyms"


## 5. Оценка фильтра

Качество проверяется на открытом наборе `dev_sentences.csv`. Положительный класс — нормализованный текст. Используются precision, recall и F1.

In [5]:
dev_df = pd.read_csv(DEV_SET, sep='|', dtype={'text': str, 'is_normalized': int})
y_true = dev_df['is_normalized'].astype(int)
y_pred = dev_df['text'].map(text_filter.filter)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
metrics_df = pd.DataFrame({
    'метрика': ['Примеров', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F1'],
    'значение': [
        len(dev_df), int(tp), int(fp), int(fn),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred),
    ],
})
display(metrics_df)
assert f1_score(y_true, y_pred) >= 0.90

,метрика,значение
0,Примеров,300.000000
1,TP,144.000000
2,FP,1.000000
3,FN,0.000000
4,Precision,0.993103
5,Recall,1.000000
6,F1,0.996540


Полученный результат на открытом наборе: **precision = 0,9931**, **recall = 1,0000**, **F1 = 0,9965**. Требуемый порог 0,90 выполнен. Единственное расхождение связано со строкой «Счёт за прошлый месяц уже сформирован.», в которой нет видимого формального дефекта; специальное правило для неё не добавлялось, чтобы не переобучаться на одной неоднозначной метке.

## 6. Обработка полного корпуса

Функция `preprocess` читает исходные метаданные, последовательно применяет нормализатор и фильтр и записывает три колонки в UTF-8. Четыре решения были уточнены ручным прослушиванием и применяются только к точным парам `id + текст`, не ослабляя общий фильтр.

In [6]:
processing_stats = preprocess(SOURCE_METADATA, NORMALIZED_METADATA)
pd.DataFrame({
    'показатель': ['Исходных строк', 'Изменено', 'Исключено', 'Сохранено', 'Проверено по аудио и сохранено'],
    'количество': [
        processing_stats['total'],
        processing_stats['changed'],
        processing_stats['rejected'],
        processing_stats['kept'],
        processing_stats['reviewed_overrides'],
    ],
})

,показатель,количество
0,Исходных строк,22200
1,Изменено,16067
2,Исключено,100
3,Сохранено,22100
4,Проверено по аудио и сохранено,4


Результат полного запуска:

| Показатель | Количество |
|---|---:|
| Исходных строк | 22 200 |
| Типографически изменено | 16 067 |
| Исключено | 100 |
| Сохранено | 22 100 |
| Сохранено после ручной проверки аудио | 4 |

Исходные WAV 44 100 Гц были предварительно преобразованы в монофонический 16-битный PCM 22 050 Гц.

In [7]:
with NORMALIZED_METADATA.open('r', encoding='utf-8', newline='') as source:
    output_rows = list(csv.reader(source, delimiter='|', quoting=csv.QUOTE_NONE))

output_ids = [row[0] for row in output_rows]
contract_checks = {
    'строк не меньше 16000': len(output_rows) >= 16_000,
    'ровно три колонки': all(len(row) == 3 for row in output_rows),
    'нормализованный текст в NFC': all(row[2] == unicodedata.normalize('NFC', row[2]) for row in output_rows),
    'идентификаторы уникальны': len(output_ids) == len(set(output_ids)),
    'для каждой строки существует WAV': all((WAV_DIR / f'{utterance_id}.wav').is_file() for utterance_id in output_ids),
}
display(pd.DataFrame(contract_checks.items(), columns=['проверка', 'результат']))
assert all(contract_checks.values())

,проверка,результат
0,строк не меньше 16000,True
1,ровно три колонки,True
2,нормализованный текст в NFC,True
3,идентификаторы уникальны,True
4,для каждой строки существует WAV,True


## 7. Ручная проверка по аудио

Неоднозначные записи были прослушаны вручную:

- `001257_RUSLAN`: `Г. П.` произнесено «гэ пэ» — запись исключена;
- `018457_RUSLAN`: `Д. Я. Р.` произнесено «дэ я эр», а `ДЯР` — «дяр» — запись исключена из-за первой части;
- `000993_RUSLAN`, `001766_RUSLAN`: аббревиатуры произнесены по буквам — исключены;
- `009017_RUSLAN`, `021885_RUSLAN`: цифры произнесены словами — исключены;
- `001028_RUSLAN`, `006729_RUSLAN`: сокращения произнесены буквально — сохранены без раскрытия;
- `000880_RUSLAN`: косые черты не произносятся — они удалены, запись сохранена;
- `001024_RUSLAN`: звёздочка не произносится — она удалена, запись сохранена.

Такой подход сохраняет точное соответствие текста и речи и не переносит частные аудио-решения на закрытый тест.

## 8. Автоматические тесты

Тесты проверяют фильтр, нормализатор, сохранение `ё`, регистра и ударения, минимальную F1, формат выходного файла и аудио-проверенные исключения.

In [8]:
test_result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', str(LAB_DIR), '-p', 'test*.py', '-v'],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(test_result.stdout)
print(test_result.stderr)
assert test_result.returncode == 0


test_applies_only_reviewed_corpus_overrides (test_lab1_text.PreprocessTests.test_applies_only_reviewed_corpus_overrides) ... ok
test_writes_exactly_three_columns (test_lab1_text.PreprocessTests.test_writes_exactly_three_columns) ... ok
test_accepts_literal_russian_text (test_lab1_text.TextFilterTests.test_accepts_literal_russian_text) ... ok
test_dev_f1_is_above_target (test_lab1_text.TextFilterTests.test_dev_f1_is_above_target) ... ok
test_rejects_non_literal_notation (test_lab1_text.TextFilterTests.test_rejects_non_literal_notation) ... ok
test_normalizes_safe_typography (test_lab1_text.TextNormalizerTests.test_normalizes_safe_typography) ... ok
test_preserves_yo_case_and_stress (test_lab1_text.TextNormalizerTests.test_preserves_yo_case_and_stress) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.043s

OK



## 9. Выводы

В работе выполнен анализ корпуса RUSLAN, разработаны интерпретируемый фильтр и консервативный нормализатор, а также сформированы итоговые метаданные для следующих этапов TTS.

Фильтр достиг F1 = 0,9965 на открытом наборе при требовании не менее 0,90. Из 22 200 фраз сохранено 22 100, что значительно превышает минимальный объём 16 000. Итоговый файл содержит ровно три колонки, записан в UTF-8, находится в Unicode NFC и не содержит дублирующихся идентификаторов или ссылок на отсутствующие аудиофайлы.

Основное ограничение решения — намеренно консервативная обработка чисел и сокращений. Их автоматическое раскрытие без контекста и аудио может нарушить выравнивание, поэтому такие записи исключаются либо проверяются вручную.